# The Streaming Data Loop

The full physical-AI data loop in one notebook: **record** a dataset into a
Hugging Face Storage Bucket, **stream** it back with no download, **train** a
policy on the streamed data, and **load** the checkpoint ready to deploy.

This is the notebook version of
[`examples/06_agent_collect_and_stream.py`](https://github.com/strands-labs/robots/blob/main/examples/06_agent_collect_and_stream.py), the agent-driven data-loop example.

**Requirements:** the bucket path needs `strands-robots >= 0.5.1`, which is the
first release whose `[lerobot]` extra floors LeRobot at the `>= 0.6.1` that
serves `stream_dataset(..., repo_type="bucket")`.

```bash
pip install -U "strands-robots[sim-mujoco,lerobot]>=0.5.1"
```

Keep the `-U` and the version floor on any install line you adapt. Extras alone
do not make a requirement unsatisfied, so a plain install into an environment
that already has an older release reports `Requirement already satisfied` and
upgrades nothing. On 0.4.1 the bucket read below then raises `TypeError: open()
got an unexpected keyword argument 'repo_type'`, naming the keyword rather than
the stale install behind it.

Bucket steps additionally need `hf auth login`. Step 5 needs
`lerobot[training]` on CPU as well as GPU - the trainer imports `accelerate`
either way. The sim-only path (record + stream from a local root) runs on any
laptop, no GPU, no credentials.

The optional Step 7 (Isaac backend swap) additionally needs an RTX GPU +
Isaac Sim 6.0+ and the `sim-isaac` extra; it self-skips otherwise.

In [ ]:
import os
import shutil
import sys

# macOS uses "cgl" for offscreen GL; Linux headless uses "egl".
os.environ.setdefault("MUJOCO_GL", "cgl" if sys.platform == "darwin" else "egl")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")

ROOT = "/tmp/nb5_dataset"
OUT = "/tmp/nb5_ft"
BUCKET = None  # set to "your-org/your-bucket" to enable the bucket path
RUN_ID = "nb5_demo"  # folder inside the bucket; one run per collection session

shutil.rmtree(ROOT, ignore_errors=True)
shutil.rmtree(OUT, ignore_errors=True)

## 1. Record a demonstration

`Robot("so100")` builds a MuJoCo simulation. We add a camera (so the dataset
has video), run the Mock policy for 60 steps (structurally complete data), and
stop recording. The result is a LeRobotDataset on disk.

In [ ]:
from strands_robots import MockPolicy, Robot, create_policy
from strands_robots.training import TrainSpec, create_trainer

sim = Robot("so100", mesh=False)
sim.add_camera(name="front", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05])
sim.start_recording(
    repo_id="local/nb5_demo",
    root=ROOT,
    fps=30,
    task="pick up the red cube",
    cameras=["front"],  # record only the declared sensor, not the implicit 'default' view
    overwrite=True,
)
# control_frequency must match the recording fps: the recorder writes one frame
# per control step with no decimation, so a 50 Hz default rollout against a 30 fps
# recording is refused rather than written at a distorted timestamp rate.
result = sim.run_policy(
    robot_name="so100",
    policy_object=MockPolicy(),
    instruction="pick up the red cube",
    n_steps=60,
    control_frequency=30.0,
)
if result.get("status") != "success":
    raise RuntimeError(f"rollout failed: {result.get('content')}")

result = sim.stop_recording()
if result.get("status") != "success":
    raise RuntimeError(f"recording failed: {result.get('content')}")
print("recorded ->", ROOT)

## 2. See what the robot recorded

Render the front camera so you can see the scene the dataset captured.

In [ ]:
from IPython.display import Image, display

# Render the sim before destroying it.
frame = sim.render(camera_name="front")
for item in frame.get("content", []):
    if isinstance(item, dict) and "image" in item:
        display(Image(data=item["image"]["source"]["bytes"], width=480))
        break
sim.destroy()
print("SO-100 arm with front camera - this is what the dataset contains.")

## 3. Sync to a Hugging Face Storage Bucket (optional)

If you have a bucket (`hf buckets create your-org/name --private` after
`hf auth login`), set `BUCKET` in cell 1. The sync uploads only the bytes that
changed (Xet deduplication), so daily re-syncs are fast.

**Skip this cell** if running without credentials (the local path still works
for Steps 4 and 5).

In [ ]:
if BUCKET:
    from strands_robots.dataset_recorder import DatasetRecorder

    # Reopen the dataset recorded in Step 1 and sync it. sync_to_bucket()
    # returns a status dict - check it instead of assuming success.
    rec = DatasetRecorder.resume("local/nb5_demo", root=ROOT)
    result = rec.sync_to_bucket(BUCKET, run_id=RUN_ID)
    if result.get("status") == "success":
        print(f"synced to bucket: {result['bucket_uri']}")
    else:
        raise RuntimeError(f"bucket sync failed: {result.get('message')}")
else:
    print("BUCKET not set - skipping sync. Local path works for training below.")

## 4. Stream the dataset back

`stream_dataset()` is the read counterpart to `start_recording()`. It reads
frames lazily with no full download. If you synced to a bucket (Step 3), pass
`repo_type="bucket"` to stream directly from it (requires LeRobot >= 0.6.1).

`sync_to_bucket` writes each run to its own `run_id` folder inside the bucket
(`hf://buckets/{BUCKET}/{RUN_ID}`), so the streaming repo id is
`f"{BUCKET}/{RUN_ID}"`, not `BUCKET` on its own. The bucket namespace is the
first two segments of that id; everything after it is the path within the
bucket. Passing `BUCKET` alone looks for `meta/` at the bucket root and fails
with `FileNotFoundError`.

In [ ]:
sim = Robot("so100", mesh=False)

if BUCKET:
    # Read from the same path Step 3 wrote to: bucket namespace + run_id.
    bucket_repo_id = f"{BUCKET}/{RUN_ID}"
    reader = sim.stream_dataset(bucket_repo_id, repo_type="bucket", shuffle=False)
    print(f"streaming from bucket: hf://buckets/{bucket_repo_id}")
else:
    reader = sim.stream_dataset("local/nb5_demo", root=ROOT, shuffle=False)
    print(f"streaming from local root: {ROOT}")

print(f"episodes: {reader.num_episodes} | frames: {reader.num_frames} | fps: {reader.fps}")

for n, frame in enumerate(reader):
    if n == 0:
        img = frame.get("observation.images.front")
        state = frame.get("observation.state")
        print(
            f"frame 0 - image: {tuple(img.shape) if img is not None else None}, state: {tuple(state.shape) if state is not None else None}"
        )
    if n >= 4:
        break
print(f"streamed {n + 1} frames - camera decoded on the fly.")
sim.destroy()

## 5. Train a policy on the dataset

`create_trainer("lerobot_local")` returns a Trainer (the peer of
`create_policy()`). A `TrainSpec` describes the run. Here we train ACT for 2
steps on CPU (fast for the demo). On a GPU with 500 steps this takes about 133
seconds on an NVIDIA L4 (`g6.4xlarge`).

This step needs `lerobot[training]` whichever device you are on: the trainer
imports `accelerate` before it looks at the device. The cell checks
`result.status` and re-raises `result.message`, which carries LeRobot's own
install remedy - `train()` reports failure in its result rather than raising, so
an unchecked call would hand Step 6 a `checkpoint_dir` of `None`.

In [ ]:
trainer = create_trainer("lerobot_local", device="cpu")
spec = TrainSpec(
    dataset_root=ROOT,
    base_model="",  # ACT from scratch
    output_dir=OUT,
    steps=2,  # raise to 500+ on a GPU for a real checkpoint
    save_freq=2,
    global_batch_size=2,
    extra={"policy_type": "act", "num_workers": 0},
)

problems = trainer.validate(spec)
assert not problems, problems

result = trainer.train(spec)
if result.status != "success":
    # train() converts any failure into a TrainResult rather than raising, and
    # result.message carries the underlying cause - including lerobot's own
    # "'accelerate' is required but not installed" remedy. Without this check the
    # cell prints status=error, and the next cell fails on a None checkpoint_dir
    # instead of on the thing that actually went wrong.
    raise RuntimeError(f"training failed: {result.message}")
print(f"train status: {result.status}")
print(f"checkpoint: {result.checkpoint_dir}")

## 6. Load the trained checkpoint

The same `create_policy()` entry point loads the checkpoint we just produced.
On hardware you would pass `mode="real"` to deploy against a physical arm.

In [ ]:
policy = create_policy(result.checkpoint_dir)
print(f"loaded: {type(policy).__name__}")
print("record -> stream -> train -> load: the data loop closes.")

## 7. Same loop, different backend: Isaac (optional - RTX GPU + Isaac Sim)

The record -> stream loop above ran on the default MuJoCo backend - no GPU, no
credentials. Backend parity for recording landed in #1552 (`IsaacRecordingMixin`),
so the *same* loop runs on NVIDIA Isaac Sim with a single `backend="isaac"` swap.

This step is optional and self-skips unless you have:

- an RTX GPU and Isaac Sim 6.0+ installed out-of-band, and
- `pip install -U 'strands-robots[sim-isaac,lerobot]'`.

One pacing difference from the MuJoCo cells above: Isaac renders at
`rendering_dt = 1/30` by default, so keep `control_frequency <= 1 / rendering_dt`.
We use `fps=10` / `control_frequency=10.0` here (not the MuJoCo cells' `fps=30`)
so every recorded frame reflects a freshly rendered product.

In [ ]:
from strands_robots.simulation.isaac import IsaacSimulation

available, reason = IsaacSimulation.is_available()
if not available:
    print(f"Isaac Sim not available - skipping the backend-swap demo: {reason}")
else:
    ROOT_ISAAC = "/tmp/nb5_isaac_dataset"
    shutil.rmtree(ROOT_ISAAC, ignore_errors=True)
    isaac = Robot("so100", backend="isaac", mesh=False)  # same loop, one kwarg changed
    isaac.add_camera(name="front", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05])
    isaac.start_recording(
        repo_id="local/nb5_isaac",
        root=ROOT_ISAAC,
        fps=10,
        task="pick up the red cube",
        cameras=["front"],
        overwrite=True,
    )
    isaac.run_policy(
        robot_name="so100",
        policy_object=MockPolicy(),
        n_steps=20,
        control_frequency=10.0,
        fast_mode=True,
        instruction="pick up the red cube",
    )
    isaac.stop_recording()
    reader = isaac.stream_dataset("local/nb5_isaac", root=ROOT_ISAAC, shuffle=False)
    print(f"isaac dataset: episodes={reader.num_episodes} frames={reader.num_frames}")
    isaac.destroy()

## Where to go from here

- Raise `steps` to 500 and run on a GPU to get a checkpoint that loads and runs.
  It will not be a good policy: this notebook records one 60-frame episode from
  the Mock policy, so what the run proves is the record-train-load path, not the
  behaviour. Collect real demonstrations before you judge a checkpoint.
- Set `BUCKET` and run with `hf auth login` to exercise the full bucket path.
- Swap `mode="real"` on `Robot("so100")` to deploy to a physical SO-101.
- Read the [first post in the series](https://huggingface.co/blog/amazon/strands-lerobot-hub-to-hardware) for the full sim-to-hardware walkthrough.
- See [examples/06_agent_collect_and_stream.py](https://github.com/strands-labs/robots/blob/main/examples/06_agent_collect_and_stream.py) for the agent-driven version.